In [1]:
import pandas as pd

df = pd.read_parquet("data/raw/yellow_tripdata_2025-01.parquet")

print("Rows and columns:", df.shape)
print(df.head().to_string())

Rows and columns: (3475226, 20)
   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  improvement_surcharge  total_amount  congestion_surcharge  Airport_fee  cbd_congestion_fee
0         1  2025-01-01 00:18:38   2025-01-01 00:26:59              1.0           1.60         1.0                  N           229           237             1         10.0    3.5      0.5        3.00           0.0                    1.0         18.00                   2.5          0.0                 0.0
1         1  2025-01-01 00:32:40   2025-01-01 00:35:13              1.0           0.50         1.0                  N           236           237             1          5.1    3.5      0.5        2.02           0.0                    1.0         12.12                   2.5          0.0                 0.0
2         1  2025-01-01 00:44:04   2025-01-01 0

In [2]:
df["duration_minutes"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df[[
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "duration_minutes",
]].head()

,tpep_pickup_datetime,tpep_dropoff_datetime,duration_minutes
0,2025-01-01 00:18:38,2025-01-01 00:26:59,8.350000
1,2025-01-01 00:32:40,2025-01-01 00:35:13,2.550000
2,2025-01-01 00:44:04,2025-01-01 00:46:01,1.950000
3,2025-01-01 00:14:27,2025-01-01 00:20:01,5.566667
4,2025-01-01 00:21:34,2025-01-01 00:25:06,3.533333


In [3]:
df["duration_minutes"].describe()

count    3.475226e+06
mean     1.501812e+01
std      3.871358e+01
min     -5.147232e+04
25%      7.283333e+00
50%      1.170000e+01
75%      1.833333e+01
max      5.626317e+03
Name: duration_minutes, dtype: float64

In [4]:
print("Missing durations:", df["duration_minutes"].isna().sum())
print("Zero or negative:", (df["duration_minutes"] <= 0).sum())
print("Over two hours:", (df["duration_minutes"] > 120).sum())

Missing durations: 0
Zero or negative: 2051
Over two hours: 2014


In [5]:
df_clean = df.loc[
    (df["duration_minutes"] > 0)
    & (df["duration_minutes"] <= 120)
].copy()

print("Entfernte Zeilen:", len(df) - len(df_clean))
print("Verbleibende Zeilen:", len(df_clean))

Entfernte Zeilen: 4065
Verbleibende Zeilen: 3471161


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

# Eingaben und Ziel vorbereiten
X = df_clean[["PULocationID", "DOLocationID"]].copy()
X["pickup_hour"] = df_clean["tpep_pickup_datetime"].dt.hour
X["pickup_weekday"] = df_clean["tpep_pickup_datetime"].dt.dayofweek
y = df_clean["duration_minutes"]

# 80 % zum Lernen, 20 % zur abschließenden Bewertung
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Gebietsnummern als Kategorien behandeln
preprocessing = ColumnTransformer(
    [
        ("locations", OneHotEncoder(handle_unknown="ignore"),
         ["PULocationID", "DOLocationID"])
    ],
    remainder="passthrough"
)

# Begrenzte Größe für ein erstes Basismodell
model = Pipeline([
    ("preprocessing", preprocessing),
    ("regressor", RandomForestRegressor(
        n_estimators=50,
        max_depth=15,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

print("Training startet …")
model.fit(X_train, y_train)

predictions = model.predict(X_test)
rmse = root_mean_squared_error(y_test, predictions)

print(f"Test-RMSE: {rmse:.2f} Minuten")

Training startet …
Test-RMSE: 8.79 Minuten


In [8]:
import numpy as np

baseline_predictions = np.full(len(y_test), y_train.mean())
baseline_rmse = root_mean_squared_error(y_test, baseline_predictions)

print(f"Immer den Mittelwert vorhersagen: {baseline_rmse:.2f} Minuten")
print(f"Random Forest:                   {rmse:.2f} Minuten")

Immer den Mittelwert vorhersagen: 11.20 Minuten
Random Forest:                   8.79 Minuten


In [9]:
from pathlib import Path
import joblib

artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

model_path = artifact_dir / "taxi_duration_pipeline.joblib"
joblib.dump(model, model_path)

print(f"Gespeichert unter: {model_path.resolve()}")

Gespeichert unter: /Users/christinck/Documents/ml_bootcamp/week-5/deployment_project/artifacts/taxi_duration_pipeline.joblib
